In [ ]:
import os
import re
import sys
import csv
import json
import torch
import pickle
import contextlib
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Dict

In [ ]:
def load_raw_ds(file_path: str, amount=None):
    file_path_segments = file_path.split(".")

    match file_path_segments[-1]:
        case "jsonl":
            if file_path.__contains__("MatchSum"):
                return load_MatchSumm_json(file_path, amount)
            return load_json(file_path, amount)

        case "pt":
            if file_path.__contains__("PreSumm"):
                return load_PreSumm_pt(file_path, amount)
            return load_pt(file_path, amount)

        case "csv":
            if file_path.__contains__("kaggle"):
                return load_kaggle_csv(file_path, amount)
            return load_csv(file_path, amount)

        # case 'txt':
        #     return load_txt(file_path, amount)

        case _:
            assert FileNotFoundError


def load_MatchSumm_json(file_path: str, amount=None):
    raw_ds = []
    with open(file_path, "r", encoding="utf-8") as file:
        for i, line in enumerate(file):
            if amount and amount == i:
                break

            loaded_data = json.loads(line)

            raw_ds.append(
                {
                    "text_sentences": loaded_data.get("text"),
                    "labels": loaded_data.get("label"),
                }
            )

    file.close()
    return raw_ds


def load_json(file_path: str, amount=None):
    raw_ds = []
    with open(file_path, "r", encoding="utf-8") as file:
        for i, line in enumerate(file):
            if amount and amount == i:
                break

            loaded_data = json.loads(line)

            text = loaded_data.get("text")
            text_sentences = loaded_data.get("text_sentences")
            summary = loaded_data.get("summary")
            summary_sentences = loaded_data.get("summary_sentences")
            labels = loaded_data.get("labels")

            if text is not None or summary is not None:
                raw_ds.append(
                    {
                        "text": text,
                        "text_sentences": text_sentences,
                        "summary": summary,
                        "summary_sentences": summary_sentences,
                        "labels": labels,
                    }
                )

            else:
                print(f"{i}. NULL INSTANCE\ntext: {text}\n{summary}\n")

    file.close()
    return raw_ds


def load_pt(file_path: str, amount=None):
    raw_ds = torch.load(file_path)

    return raw_ds


def load_PreSumm_pt(file_path: str, amount=None):
    raw_ds = []
    loaded_data = torch.load(file_path)

    for i, line in enumerate(loaded_data):
        if amount and amount == i:
            break

        raw_ds.append(
            {
                "text_sentences": line["src_txt"],
                "labels": line["src_sent_labels"],
            }
        )
    return raw_ds


def load_csv(file_path: str, amount=None):
    with open(file_path, mode="r") as file:
        csvFile = csv.reader(file)

    return csvFile


def load_kaggle_csv(file_path: str, amount=None):
    raw_ds = []
    with open(file_path, mode="r") as file:
        loaded_data = csv.reader(file)
        for i, line in enumerate(loaded_data):
            raw_ds.append(
                {
                    "text_sentences":   line[1],
                    "labels":           line[2],
                }
            )

            if i == 2:
                break

    return raw_ds[1:]

In [ ]:
#################################################
####         Encoded data analysation        ####
#################################################

def get_deep_type(obj):
    if isinstance(obj, list):
        # We look at the unique types inside the list to keep it readable
        inner_types = {get_deep_type(item) for item in obj}
        return f"List[{' | '.join(sorted(inner_types))}]"

    elif isinstance(obj, dict):
        # We summarize the types of all keys and all values
        key_types = {get_deep_type(k) for k in obj.keys()}
        val_types = {get_deep_type(v) for v in obj.values()}
        return f"Dict[{' | '.join(sorted(key_types))}, {' | '.join(sorted(val_types))}]"

    else:
        # Return the class name (e.g., 'Diagram' or 'str')
        return type(obj).__name__

def find_mismatches(data_a, data_b):
    # 1. Check if the outer lists are even the same length
    if len(data_a) != len(data_b):
        print(
            f"❌ [CRITICAL] Outer List Length Mismatch: List A={len(data_a)}, List B={len(data_b)}"
        )

    # Iterate through the top-level list
    for i, (dict_a, dict_b) in enumerate(zip(data_a, data_b)):
        # Check if the keys in the dictionaries match
        keys_a = set(dict_a.keys())
        keys_b = set(dict_b.keys())

        if keys_a != keys_b:
            print(f"❌ [Index {i}] Key Mismatch:")
            print(f"   Keys only in A: {keys_a - keys_b}")
            print(f"   Keys only in B: {keys_b - keys_a}")
            continue  # Skip to next list item if keys don't match

        # Check values for each key
        for key in keys_a:
            list_a = dict_a[key]
            list_b = dict_b[key]

            # 2. Check lengths of the lists inside the dictionary
            if len(list_a) != len(list_b):
                print(f"❌ [Index {i}][Key: '{key}'] Inner List Length Mismatch:")
                print(f"   Length A: {len(list_a)}")
                print(f"   Length B: {len(list_b)}")

            # 3. Check individual elements inside those lists
            # This handles List[Diagram], List[List[int]], and List[str]
            for j, (val_a, val_b) in enumerate(zip(list_a, list_b)):
                if val_a != val_b:
                    print(
                        f"❌ [Index {i}][Key: '{key}'][Inner Index {j}] Content Mismatch!"
                    )
                    print(f"   Type A: {type(val_a).__name__}")
                    print(f"   Type B: {type(val_b).__name__}")

                    # If they are small (like List[int] or str), print the actual value
                    if not hasattr(
                        val_a, "draw"
                    ):  # Don't print full Diagrams, they are too big
                        print(f"   Value A: {val_a}")
                        print(f"   Value B: {val_b}")
                    else:
                        print(
                            f"   (Diagram content differs - possibly different boxes or wires)"
                        )

In [ ]:
# raw_ds = load_raw_ds("Dataset/Raw/cnn_dailymail/train.jsonl")
# raw_ds = load_raw_ds("Dataset/Raw/cnn_dailymail/_PreSumm/cnndm.valid.0.bert.pt")
# raw_ds = load_raw_ds("Dataset/Raw/cnn_dailymail/_MatchSum/val_CNNDM_bert.jsonl")
raw_ds = load_raw_ds("Dataset/Raw/cnn_dailymail/_kaggle/validation.csv")

In [ ]:
print(get_deep_type(raw_ds))
print(raw_ds[0].keys())

for i, e in enumerate(raw_ds[0]['text_sentences']):
    print(f"{i}: {e}")

print()

a = []
for i, e in enumerate(raw_ds[0]['labels']):
    if e == 1:
        a.append(i)

print(a)



In [ ]:
def print_labels(ds, k):
    print(ds[k]["labels"])


def print_sentences(ds, k):
    x = ds[k]["text_sentences"]
    for i, e in enumerate(x):
        if e is None:
            print(f"None value in {i} element")
            continue
        print(f"{i}: {e}")

k = 0
print_labels(raw_ds, k)
print_sentences(raw_ds, k)